# WSi vs WSi + SiPM ZDC vertex reconstruction

This notebook is organized so the comparison is explicit and reproducible:

1. Load truth and detector hits.
2. Encode WSi and SiPM separately.
3. Train two models with the same event split: **WSi only** and **WSi + SiPM**.
4. Evaluate both models on the same held-out test set.
5. Produce compact comparison plots and a metrics table.

The SiPM implementation rasterizes hits in global detector coordinates. That means the layer-offset geometry is represented by the actual hit positions instead of being forced into an artificial un-offset tile index map.


In [1]:
import copy
from dataclasses import dataclass
from pathlib import Path

import awkward as ak
import numpy as np
import uproot as up
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## Configuration

In [2]:
PHOTON1_FILE = "../HEPMC-testing/photon1_test_50000.root"
PHOTON2_FILE = "../HEPMC-testing/photon2_test_50000.root"
NEUTRON_FILE = "../HEPMC-testing/neutron_test_50000.root"
TREE_NAME = "events"

# Coordinate convention
ZDC_FACE_Z_MM = 35856.4
ZDC_HALF_WIDTH_X_MM = 300.0
ZDC_HALF_WIDTH_Y_MM = 300.0
ROTATE_TO_ZDC_FRAME = True
ROTATION_THETA_RAD = 0.025

# Particle masses [GeV]
M_GAMMA = 0.0
M_NEUTRON = 0.939565
M_PI0 = 0.134977
M_LAMBDA = 1.115683

# If your SiPM collection has a different name, add it here.
SIPM_COLLECTION_CANDIDATES = [
    # Current ePIC SiPM-on-tile ZDC HCal collection
    "HcalFarForwardZDCHits",

    # Older/local names that may appear in private geometry studies
    "ZDC_SiPM_Hits",
    "ZDC_HcalSiPM_Hits",
    "ZDC_ScFi_Hits",
    "ZDC_HCalHits",
    "ZDC_Hcal_Hits",
    "ZDC_SciHits",
]

@dataclass(frozen=True)
class ImageConfig:
    n_layers: int
    nx: int
    ny: int
    x_min: float
    x_max: float
    y_min: float
    y_max: float

WSI_CFG = ImageConfig(
    n_layers=20, nx=60, ny=60,
    x_min=-300.0, x_max=300.0,
    y_min=-300.0, y_max=300.0,
)

# Coarse global rasterization of the SiPM HCal. This keeps the offset scheme implicitly,
# because hits are binned by their actual global x/y coordinates.
SIPM_CFG = ImageConfig(
    n_layers=64, nx=11, ny=12,
    x_min=-288.0, x_max=288.0,
    y_min=-294.0, y_max=294.0,
)


## Geometry and branch helpers

In [3]:

WSI_FRONT_Z_MM = 35856.4
ZDC_TUNGSTEN_THICKNESS_MM = 3.5
ZDC_GLUE_THICKNESS_MM = 0.11
ZDC_PAD_THICKNESS_MM = 0.320
ZDC_PCB_THICKNESS_MM = 1.6
ZDC_SI_AIR_THICKNESS_MM = 1.0

WSI_LAYER_THICKNESS_MM = (
    ZDC_TUNGSTEN_THICKNESS_MM
    + ZDC_GLUE_THICKNESS_MM
    + ZDC_PAD_THICKNESS_MM
    + ZDC_GLUE_THICKNESS_MM
    + ZDC_PCB_THICKNESS_MM
    + ZDC_SI_AIR_THICKNESS_MM
)

WSI_SI_CENTER_OFFSET_MM = (
    ZDC_TUNGSTEN_THICKNESS_MM
    + ZDC_GLUE_THICKNESS_MM
    + 0.5 * ZDC_PAD_THICKNESS_MM
)

WSI_LAYER_Z_MM = (
    WSI_FRONT_Z_MM
    + WSI_SI_CENTER_OFFSET_MM
    + np.arange(WSI_CFG.n_layers) * WSI_LAYER_THICKNESS_MM
)

# SiPM-on-tile HCal geometry from the ePIC compact constants.
# Values are converted to mm.
SIPM_AIR_THICKNESS_MM = 1.212
SIPM_ABSORBER_THICKNESS_MM = 20.0
SIPM_POLYSTYRENE_THICKNESS_MM = 4.0
SIPM_PCB_THICKNESS_MM = 1.6
SIPM_ESR_FOIL_THICKNESS_MM = 0.15
SIPM_BACKPLATE_THICKNESS_MM = 20.0
SIPM_FRONT_FACE_Z_MM = 35800.0
SIPM_N_LAYERS = 64

SIPM_LAYER_THICKNESS_MM = (
    SIPM_ABSORBER_THICKNESS_MM
    + SIPM_POLYSTYRENE_THICKNESS_MM
    + SIPM_PCB_THICKNESS_MM
    + 2.0 * SIPM_ESR_FOIL_THICKNESS_MM
    + 2.0 * SIPM_AIR_THICKNESS_MM
)

SIPM_LENGTH_MM = SIPM_N_LAYERS * SIPM_LAYER_THICKNESS_MM + SIPM_BACKPLATE_THICKNESS_MM

# Approximate scintillator center within one repeated steel layer. This is only used for
# diagnostics/plots; layer assignment below uses the full repeated-layer interval.
SIPM_SCINT_CENTER_OFFSET_MM = (
    SIPM_ABSORBER_THICKNESS_MM
    + SIPM_AIR_THICKNESS_MM
    + SIPM_ESR_FOIL_THICKNESS_MM
    + 0.5 * SIPM_POLYSTYRENE_THICKNESS_MM
)

SIPM_SCINT_CENTER_Z_MM = (
    SIPM_FRONT_FACE_Z_MM
    + SIPM_SCINT_CENTER_OFFSET_MM
    + np.arange(SIPM_CFG.n_layers) * SIPM_LAYER_THICKNESS_MM
)


def rotate_vectors_numpy(x, y, z, theta=0.025):
    c = np.cos(theta)
    s = np.sin(theta)
    return c * x + s * z, y, -s * x + c * z


def first_per_event(arr):
    return np.array([x[1] if len(x) > 1 else np.nan for x in arr])


def list_hit_collections(filename):
    tree = up.open(filename)[TREE_NAME]
    return sorted({k.split(".position.x")[0] for k in tree.keys() if k.endswith(".position.x")})


def collection_key_variants(name):
    """Return possible EDM4hep branch prefixes for a logical collection name.

    Depending on how the ROOT file was written, a collection may appear either as
    ``HcalFarForwardZDCHits.position.x`` or as the nested prefix
    ``HcalFarForwardZDCHits/HcalFarForwardZDCHits.position.x``.
    """
    variants = [name]
    if "/" not in name:
        variants.append(f"{name}/{name}")
    return variants


def choose_hit_collection(filename, candidates):
    tree = up.open(filename)[TREE_NAME]
    keys = set(tree.keys())
    tried = []
    for name in candidates:
        for prefix in collection_key_variants(name):
            tried.append(prefix)
            if f"{prefix}.position.x" in keys and f"{prefix}.energy" in keys:
                return prefix
    available = list_hit_collections(filename)
    raise KeyError(
        "Could not find a SiPM hit collection. "
        f"Tried {tried}. Available hit-like collections: {available}"
    )


def event_has_valid_activity(*particle_hit_blocks, require_positive_energy=True):
    """Return an event mask requiring at least one valid hit across particles.

    Each particle block is a dictionary with ``layer`` and ``E`` object arrays, one entry
    per event. This is intentionally an event-level requirement: the event is kept if
    the detector has any usable activity from gamma1, gamma2, or neutron.
    """
    n_events = len(particle_hit_blocks[0]["E"])
    keep = np.zeros(n_events, dtype=bool)

    for block in particle_hit_blocks:
        for i, (layers, energies) in enumerate(zip(block["layer"], block["E"])):
            layers = np.asarray(layers)
            energies = np.asarray(energies)
            if len(layers) == 0 or len(energies) == 0:
                continue
            valid = layers >= 0
            if require_positive_energy:
                valid = valid & (energies > 0)
            if np.any(valid):
                keep[i] = True
    return keep


def apply_event_mask_to_hit_tuple(hit_tuple, keep):
    return [arr[keep] for arr in hit_tuple]


def apply_event_mask_to_layers(layers, keep):
    return layers[keep]


def assign_layers_from_geometry(z_column, layer_z, tolerance=1.0):
    all_layers = []
    for zevt in z_column:
        zevt = np.asarray(zevt)
        if len(zevt) == 0:
            all_layers.append(np.array([], dtype=np.int32))
            continue
        dz = np.abs(zevt[:, None] - layer_z[None, :])
        layers = np.argmin(dz, axis=1)
        min_dz = dz[np.arange(len(zevt)), layers]
        out = np.full(len(zevt), -1, dtype=np.int32)
        out[min_dz < tolerance] = layers[min_dz < tolerance]
        all_layers.append(out)
    return np.array(all_layers, dtype=object)


def assign_sipm_layers_from_geometry(z_column, front_z_mm=SIPM_FRONT_FACE_Z_MM):
    """Assign SiPM HCal layer from local detector-frame z.

    HcalFarForwardZDCHits does not provide an explicit layer index. After the optional
    rotation into the ZDC frame, each hit is assigned by

        layer = floor((z - front_face_z) / single_layer_thickness)

    The final steel backplate is intentionally excluded, so anything outside the 64
    repeated layers is marked -1.
    """
    if not ROTATE_TO_ZDC_FRAME:
        print(
            "Warning: ROTATE_TO_ZDC_FRAME is False. SiPM layer assignment is most reliable "
            "after rotating global positions into the ZDC frame, because the detector is "
            "rotated by the crossing angle."
        )

    all_layers = []
    for zevt in z_column:
        zevt = np.asarray(zevt)
        if len(zevt) == 0:
            all_layers.append(np.array([], dtype=np.int32))
            continue

        layer_float = (zevt - front_z_mm) / SIPM_LAYER_THICKNESS_MM
        layers = np.floor(layer_float).astype(np.int32)

        valid = (layers >= 0) & (layers < SIPM_CFG.n_layers)
        out = np.full(len(zevt), -1, dtype=np.int32)
        out[valid] = layers[valid]
        all_layers.append(out)

    return np.array(all_layers, dtype=object)


def print_sipm_layer_diagnostics(*layer_arrays):
    layers = np.concatenate([
        np.asarray(ev_layers, dtype=np.int32)
        for arr in layer_arrays
        for ev_layers in arr
        if len(ev_layers) > 0
    ])
    if len(layers) == 0:
        print("SiPM layer diagnostic: no hits found.")
        return

    n_bad = np.count_nonzero(layers < 0)
    valid = layers[layers >= 0]
    if len(valid) == 0:
        print(f"SiPM layer diagnostic: all {len(layers)} hits were outside the expected layer range.")
        return

    counts = np.bincount(valid, minlength=SIPM_CFG.n_layers)
    occupied = np.flatnonzero(counts > 0)
    print(
        "SiPM layer diagnostic: "
        f"{len(valid)} valid hits, {n_bad} out-of-range hits, "
        f"occupied layer range {occupied.min()}–{occupied.max()}."
    )


## Loading truth and detector hits

In [4]:
def load_first_mc_particle(filename, mass_assumption):
    tree = up.open(filename)[TREE_NAME]

    vx = first_per_event(tree["MCParticles.vertex.x"].array(library="np"))
    vy = first_per_event(tree["MCParticles.vertex.y"].array(library="np"))
    vz = first_per_event(tree["MCParticles.vertex.z"].array(library="np"))

    px = first_per_event(tree["MCParticles.momentum.x"].array(library="np"))
    py = first_per_event(tree["MCParticles.momentum.y"].array(library="np"))
    pz = first_per_event(tree["MCParticles.momentum.z"].array(library="np"))

    if ROTATE_TO_ZDC_FRAME:
        vx, vy, vz = rotate_vectors_numpy(vx, vy, vz, ROTATION_THETA_RAD)
        px, py, pz = rotate_vectors_numpy(px, py, pz, ROTATION_THETA_RAD)

    p = np.sqrt(px**2 + py**2 + pz**2)
    E = np.sqrt(p**2 + mass_assumption**2)
    mass = np.full_like(px, mass_assumption, dtype=float)

    return {"vx": vx, "vy": vy, "vz": vz, "px": px, "py": py, "pz": pz, "p": p, "E": E, "m": mass}


def extract_hit_information(filename, collection_name):
    tree = up.open(filename)[TREE_NAME]
    hit_x = tree[f"{collection_name}.position.x"].array(library="np")
    hit_y = tree[f"{collection_name}.position.y"].array(library="np")
    hit_z = tree[f"{collection_name}.position.z"].array(library="np")
    hit_E = tree[f"{collection_name}.energy"].array(library="np")

    if ROTATE_TO_ZDC_FRAME:
        hit_x, hit_y, hit_z = rotate_vectors_numpy(hit_x, hit_y, hit_z, ROTATION_THETA_RAD)

    return hit_x, hit_y, hit_z, hit_E


def project_to_zdc(p):
    with np.errstate(divide="ignore", invalid="ignore"):
        dz = ZDC_FACE_Z_MM - p["vz"]
        x_zdc = p["vx"] + (p["px"] / p["pz"]) * dz
        y_zdc = p["vy"] + (p["py"] / p["pz"]) * dz
    return x_zdc, y_zdc


def intersects_zdc(x, y):
    return (
        np.isfinite(x) & np.isfinite(y)
        & (np.abs(x) < ZDC_HALF_WIDTH_X_MM)
        & (np.abs(y) < ZDC_HALF_WIDTH_Y_MM)
    )


def filter_accepted_events(filename_gamma1, filename_gamma2, filename_neutron):
    g1 = load_first_mc_particle(filename_gamma1, M_GAMMA)
    g2 = load_first_mc_particle(filename_gamma2, M_GAMMA)
    n = load_first_mc_particle(filename_neutron, M_NEUTRON)

    x1, y1 = project_to_zdc(g1)
    x2, y2 = project_to_zdc(g2)
    xn, yn = project_to_zdc(n)

    acc1 = intersects_zdc(x1, y1)
    acc2 = intersects_zdc(x2, y2)
    accn = intersects_zdc(xn, yn)
    forward = (g1["pz"] > 0) & (g2["pz"] > 0) & (n["pz"] > 0)

    return acc1 & acc2 & accn & forward


## Image encoders

In [5]:
def make_image_stack(hit_x, hit_y, hit_layer, hit_E, cfg):
    img = np.zeros((cfg.n_layers, cfg.nx, cfg.ny), dtype=np.float32)
    if len(hit_x) == 0:
        return img

    hit_x = np.asarray(hit_x)
    hit_y = np.asarray(hit_y)
    hit_layer = np.asarray(hit_layer, dtype=np.int32)
    hit_E = np.asarray(hit_E)

    ix = np.floor((hit_x - cfg.x_min) / (cfg.x_max - cfg.x_min) * cfg.nx).astype(np.int32)
    iy = np.floor((hit_y - cfg.y_min) / (cfg.y_max - cfg.y_min) * cfg.ny).astype(np.int32)

    mask = (
        (hit_layer >= 0) & (hit_layer < cfg.n_layers)
        & (ix >= 0) & (ix < cfg.nx)
        & (iy >= 0) & (iy < cfg.ny)
        & np.isfinite(hit_E)
    )

    np.add.at(img, (hit_layer[mask], ix[mask], iy[mask]), hit_E[mask])
    return img


def encode_particle_block(particle_hits, cfg):
    return make_image_stack(
        particle_hits["x"], particle_hits["y"], particle_hits["layer"], particle_hits["E"], cfg
    )


def log_energy_transform(x, scale=1e6):
    return np.log1p(scale * x)


## Event assembly with WSi and SiPM

The SiPM hit collection may appear either as `HcalFarForwardZDCHits.position.*` or as the nested EDM4hep-style prefix `HcalFarForwardZDCHits/HcalFarForwardZDCHits.position.*`; the helper now detects both.

The SiPM hit collection does not carry an explicit layer branch. The notebook therefore infers the layer from the hit position after rotating into the ZDC frame:

In [6]:
def build_events():
    geometric_mask = filter_accepted_events(PHOTON1_FILE, PHOTON2_FILE, NEUTRON_FILE)
    print(f"Geometrically accepted events: {geometric_mask.sum()} / {len(geometric_mask)}")

    gamma_1 = load_first_mc_particle(PHOTON1_FILE, M_GAMMA)
    gamma_2 = load_first_mc_particle(PHOTON2_FILE, M_GAMMA)
    neutron = load_first_mc_particle(NEUTRON_FILE, M_NEUTRON)

    # WSi
    wsi_g1 = extract_hit_information(PHOTON1_FILE, "ZDC_WSi_Hits")
    wsi_g2 = extract_hit_information(PHOTON2_FILE, "ZDC_WSi_Hits")
    wsi_n = extract_hit_information(NEUTRON_FILE, "ZDC_WSi_Hits")

    wsi_g1 = [a[geometric_mask] for a in wsi_g1]
    wsi_g2 = [a[geometric_mask] for a in wsi_g2]
    wsi_n = [a[geometric_mask] for a in wsi_n]

    wsi_layers_g1 = assign_layers_from_geometry(wsi_g1[2], WSI_LAYER_Z_MM, tolerance=1.0)
    wsi_layers_g2 = assign_layers_from_geometry(wsi_g2[2], WSI_LAYER_Z_MM, tolerance=1.0)
    wsi_layers_n = assign_layers_from_geometry(wsi_n[2], WSI_LAYER_Z_MM, tolerance=1.0)

    wsi_blocks_for_mask = [
        {"layer": wsi_layers_g1, "E": wsi_g1[3]},
        {"layer": wsi_layers_g2, "E": wsi_g2[3]},
        {"layer": wsi_layers_n, "E": wsi_n[3]},
    ]
    has_wsi = event_has_valid_activity(*wsi_blocks_for_mask)

    # SiPM. The collection name differs across geometry versions, so detect it.
    sipm_collection = choose_hit_collection(PHOTON1_FILE, SIPM_COLLECTION_CANDIDATES)
    print(f"Using SiPM collection: {sipm_collection}")

    sipm_g1 = extract_hit_information(PHOTON1_FILE, sipm_collection)
    sipm_g2 = extract_hit_information(PHOTON2_FILE, sipm_collection)
    sipm_n = extract_hit_information(NEUTRON_FILE, sipm_collection)

    sipm_g1 = [a[geometric_mask] for a in sipm_g1]
    sipm_g2 = [a[geometric_mask] for a in sipm_g2]
    sipm_n = [a[geometric_mask] for a in sipm_n]

    sipm_layers_g1 = assign_sipm_layers_from_geometry(sipm_g1[2])
    sipm_layers_g2 = assign_sipm_layers_from_geometry(sipm_g2[2])
    sipm_layers_n = assign_sipm_layers_from_geometry(sipm_n[2])
    print_sipm_layer_diagnostics(sipm_layers_g1, sipm_layers_g2, sipm_layers_n)

    sipm_blocks_for_mask = [
        {"layer": sipm_layers_g1, "E": sipm_g1[3]},
        {"layer": sipm_layers_g2, "E": sipm_g2[3]},
        {"layer": sipm_layers_n, "E": sipm_n[3]},
    ]
    has_sipm = event_has_valid_activity(*sipm_blocks_for_mask)

    detector_mask = has_wsi & has_sipm
    print(
        "Detector-hit requirement: "
        f"WSi {has_wsi.sum()} / {len(has_wsi)}, "
        f"SiPM {has_sipm.sum()} / {len(has_sipm)}, "
        f"both {detector_mask.sum()} / {len(detector_mask)}."
    )
    print(f"Rejected after geometric acceptance: {np.count_nonzero(~detector_mask)} events")

    # Apply the detector-hit requirement to the already geometrically accepted arrays.
    decay_x = gamma_1["vx"][geometric_mask][detector_mask]
    decay_y = gamma_1["vy"][geometric_mask][detector_mask]
    decay_z = gamma_1["vz"][geometric_mask][detector_mask]

    lam_px = (gamma_1["px"][geometric_mask] + gamma_2["px"][geometric_mask] + neutron["px"][geometric_mask])[detector_mask]
    lam_py = (gamma_1["py"][geometric_mask] + gamma_2["py"][geometric_mask] + neutron["py"][geometric_mask])[detector_mask]
    lam_pz = (gamma_1["pz"][geometric_mask] + gamma_2["pz"][geometric_mask] + neutron["pz"][geometric_mask])[detector_mask]

    wsi_g1 = apply_event_mask_to_hit_tuple(wsi_g1, detector_mask)
    wsi_g2 = apply_event_mask_to_hit_tuple(wsi_g2, detector_mask)
    wsi_n = apply_event_mask_to_hit_tuple(wsi_n, detector_mask)
    wsi_layers_g1 = apply_event_mask_to_layers(wsi_layers_g1, detector_mask)
    wsi_layers_g2 = apply_event_mask_to_layers(wsi_layers_g2, detector_mask)
    wsi_layers_n = apply_event_mask_to_layers(wsi_layers_n, detector_mask)

    sipm_g1 = apply_event_mask_to_hit_tuple(sipm_g1, detector_mask)
    sipm_g2 = apply_event_mask_to_hit_tuple(sipm_g2, detector_mask)
    sipm_n = apply_event_mask_to_hit_tuple(sipm_n, detector_mask)
    sipm_layers_g1 = apply_event_mask_to_layers(sipm_layers_g1, detector_mask)
    sipm_layers_g2 = apply_event_mask_to_layers(sipm_layers_g2, detector_mask)
    sipm_layers_n = apply_event_mask_to_layers(sipm_layers_n, detector_mask)

    events = []
    for i in range(len(decay_z)):
        events.append({
            "wsi": {
                "gamma1": {"x": wsi_g1[0][i], "y": wsi_g1[1][i], "layer": wsi_layers_g1[i], "E": wsi_g1[3][i]},
                "gamma2": {"x": wsi_g2[0][i], "y": wsi_g2[1][i], "layer": wsi_layers_g2[i], "E": wsi_g2[3][i]},
                "neutron": {"x": wsi_n[0][i], "y": wsi_n[1][i], "layer": wsi_layers_n[i], "E": wsi_n[3][i]},
            },
            "sipm": {
                "gamma1": {"x": sipm_g1[0][i], "y": sipm_g1[1][i], "layer": sipm_layers_g1[i], "E": sipm_g1[3][i]},
                "gamma2": {"x": sipm_g2[0][i], "y": sipm_g2[1][i], "layer": sipm_layers_g2[i], "E": sipm_g2[3][i]},
                "neutron": {"x": sipm_n[0][i], "y": sipm_n[1][i], "layer": sipm_layers_n[i], "E": sipm_n[3][i]},
            },
            "x_vertex": decay_x[i],
            "y_vertex": decay_y[i],
            "z_vertex": decay_z[i],
            "lam_mom_x": lam_px[i],
            "lam_mom_y": lam_py[i],
            "lam_mom_z": lam_pz[i],
        })
    return events

# Run this after confirming PHOTON1_FILE/PHOTON2_FILE/NEUTRON_FILE point to the intended ROOT files.
events = build_events()


Geometrically accepted events: 29825 / 50000
Using SiPM collection: HcalFarForwardZDCHits/HcalFarForwardZDCHits
SiPM layer diagnostic: 69156138 valid hits, 1004037 out-of-range hits, occupied layer range 5–63.
Detector-hit requirement: WSi 29825 / 29825, SiPM 29825 / 29825, both 29825 / 29825.
Rejected after geometric acceptance: 0 events


## Dataset and model definitions

In [7]:
class LambdaZDCDataset(Dataset):
    def __init__(self, events, use_sipm=False, z_mean=None, z_std=None):
        self.events = events
        self.use_sipm = use_sipm

        z_values = np.array([ev["z_vertex"] for ev in events], dtype=np.float32)
        self.z_mean = float(z_values.mean()) if z_mean is None else float(z_mean)
        self.z_std = float(z_values.std()) if z_std is None else float(z_std)
        if self.z_std == 0:
            raise ValueError("z_std is zero; target normalization is ill-defined.")

    def __len__(self):
        return len(self.events)

    def _encode_detector(self, ev, detector_name, cfg):
        imgs = []
        for particle in ["gamma1", "gamma2", "neutron"]:
            imgs.append(encode_particle_block(ev[detector_name][particle], cfg))
        x = np.stack(imgs, axis=0)  # [particles=3, layers, nx, ny]
        return log_energy_transform(x)

    def __getitem__(self, idx):
        ev = self.events[idx]
        wsi = self._encode_detector(ev, "wsi", WSI_CFG)
        z = np.float32((ev["z_vertex"] - self.z_mean) / self.z_std)

        if not self.use_sipm:
            return {
                "wsi": torch.tensor(wsi, dtype=torch.float32),
                "z": torch.tensor([z], dtype=torch.float32),
            }

        sipm = self._encode_detector(ev, "sipm", SIPM_CFG)
        return {
            "wsi": torch.tensor(wsi, dtype=torch.float32),
            "sipm": torch.tensor(sipm, dtype=torch.float32),
            "z": torch.tensor([z], dtype=torch.float32),
        }


class ZDCImageEncoder3D(nn.Module):
    def __init__(self, out_features=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(32),
            nn.SiLU(),
            nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(64),
            nn.SiLU(),
            nn.MaxPool3d(kernel_size=(1, 2, 2)),
            nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(128),
            nn.SiLU(),
            nn.MaxPool3d(kernel_size=(2, 2, 2)),
            nn.Conv3d(128, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(128),
            nn.SiLU(),
            nn.AdaptiveAvgPool3d((4, 4, 4)),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4 * 4, out_features),
            nn.SiLU(),
        )

    def forward(self, x):
        return self.net(x)


class ZDCVertexModel(nn.Module):
    def __init__(self, use_sipm=False, feature_dim=128):
        super().__init__()
        self.use_sipm = use_sipm
        self.wsi_encoder = ZDCImageEncoder3D(out_features=feature_dim)
        if use_sipm:
            self.sipm_encoder = ZDCImageEncoder3D(out_features=feature_dim)
            regressor_in = 2 * feature_dim
        else:
            self.sipm_encoder = None
            regressor_in = feature_dim

        self.regressor = nn.Sequential(
            nn.Linear(regressor_in, 128),
            nn.SiLU(),
            nn.Dropout(p=0.10),
            nn.Linear(128, 64),
            nn.SiLU(),
            nn.Linear(64, 1),
        )

    def forward(self, batch):
        wsi_features = self.wsi_encoder(batch["wsi"])
        if not self.use_sipm:
            return self.regressor(wsi_features)
        sipm_features = self.sipm_encoder(batch["sipm"])
        features = torch.cat([wsi_features, sipm_features], dim=1)
        return self.regressor(features)


## Train / evaluate utilities

In [8]:
def move_batch_to_device(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}


def make_loss_function(loss_name="mae", smooth_l1_beta=0.5):
    if loss_name == "mae":
        return nn.L1Loss()
    if loss_name == "smooth_l1":
        return nn.SmoothL1Loss(beta=smooth_l1_beta)
    if loss_name == "mse":
        return nn.MSELoss()
    raise ValueError(f"Unknown loss_name={loss_name!r}")


def warmup_cosine_scheduler(optimizer, n_epochs, warmup_epochs=5, min_lr_fraction=0.05):
    warmup_epochs = max(1, int(warmup_epochs))
    decay_epochs = max(1, int(n_epochs) - warmup_epochs)

    def lr_lambda(epoch_idx):
        if epoch_idx < warmup_epochs:
            return float(epoch_idx + 1) / float(warmup_epochs)
        progress = float(epoch_idx + 1 - warmup_epochs) / float(decay_epochs)
        cosine = 0.5 * (1.0 + np.cos(np.pi * min(1.0, progress)))
        return min_lr_fraction + (1.0 - min_lr_fraction) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def train_model_variant(
    train_events,
    val_events,
    use_sipm=False,
    n_epochs=120,
    batch_size=64,
    lr=1e-3,
    wd=1e-4,
    patience=20,
    warmup_epochs=5,
    loss_name="mae",
):
    train_dataset = LambdaZDCDataset(train_events, use_sipm=use_sipm)
    val_dataset = LambdaZDCDataset(
        val_events,
        use_sipm=use_sipm,
        z_mean=train_dataset.z_mean,
        z_std=train_dataset.z_std,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = ZDCVertexModel(use_sipm=use_sipm).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = warmup_cosine_scheduler(optimizer, n_epochs=n_epochs, warmup_epochs=warmup_epochs)
    loss_fn = make_loss_function(loss_name)

    best_val_loss = float("inf")
    best_state = None
    bad_epochs = 0
    history = []

    label = "WSi + SiPM" if use_sipm else "WSi only"
    print(f"Training {label} on {device}")

    for epoch in range(1, n_epochs + 1):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            batch = move_batch_to_device(batch, device)
            optimizer.zero_grad()
            pred = model(batch)
            loss = loss_fn(pred, batch["z"])
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch["z"].size(0)
        train_loss /= len(train_dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                batch = move_batch_to_device(batch, device)
                pred = model(batch)
                loss = loss_fn(pred, batch["z"])
                val_loss += loss.item() * batch["z"].size(0)
        val_loss /= len(val_dataset)

        current_lr = optimizer.param_groups[0]["lr"]
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "lr": current_lr})
        print(f"{label:10s} | epoch {epoch:03d} | lr {current_lr:.2e} | train {train_loss:.5f} | val {val_loss:.5f}")
        scheduler.step()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping {label} at epoch {epoch}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return {
        "model": model,
        "z_mean": train_dataset.z_mean,
        "z_std": train_dataset.z_std,
        "history": history,
        "use_sipm": use_sipm,
        "label": label,
    }


def evaluate_model(result, test_events, batch_size=128):
    dataset = LambdaZDCDataset(
        test_events,
        use_sipm=result["use_sipm"],
        z_mean=result["z_mean"],
        z_std=result["z_std"],
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = result["model"].to(device)
    model.eval()

    z_true_all = []
    z_pred_all = []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch_to_device(batch, device)
            pred_norm = model(batch).cpu().numpy().flatten()
            true_norm = batch["z"].cpu().numpy().flatten()
            z_pred_all.append(pred_norm * result["z_std"] + result["z_mean"])
            z_true_all.append(true_norm * result["z_std"] + result["z_mean"])

    z_true = np.concatenate(z_true_all)
    z_pred = np.concatenate(z_pred_all)
    residual = z_pred - z_true
    return {"label": result["label"], "z_true": z_true, "z_pred": z_pred, "residual": residual}


def summarize_evaluation(eval_result):
    r = eval_result["residual"]
    return {
        "model": eval_result["label"],
        "bias_mm": float(np.mean(r)),
        "mae_mm": float(np.mean(np.abs(r))),
        "rmse_mm": float(np.sqrt(np.mean(r**2))),
        "std_mm": float(np.std(r)),
        "q68_abs_mm": float(np.quantile(np.abs(r), 0.68)),
        "q95_abs_mm": float(np.quantile(np.abs(r), 0.95)),
    }


## Split once, train both models

In [9]:
train_events, test_events = train_test_split(events, test_size=0.15, random_state=SEED, shuffle=True)
train_events, val_events = train_test_split(train_events, test_size=0.1765, random_state=SEED, shuffle=True)

wsi_result = train_model_variant(
    train_events,
    val_events,
    use_sipm=False,
    n_epochs=120,
    batch_size=64,
    lr=1e-3,
    wd=1e-4,
    patience=20,
    warmup_epochs=5,
    loss_name="mae",
)


Training WSi only on cuda


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.10 GiB. GPU 0 has a total capacity of 3.81 GiB of which 903.00 MiB is free. Including non-PyTorch memory, this process has 2.92 GiB memory in use. Of the allocated memory 2.81 GiB is allocated by PyTorch, and 20.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## Evaluate on the same held-out test set

In [ ]:
eval_wsi = evaluate_model(wsi_result, test_events)
evaluations = [eval_wsi]

metrics = [summarize_evaluation(e) for e in evaluations]
for row in metrics:
    print(row)


## Comparison plots

In [ ]:
def lambda_momentum_arrays(events):
    px = np.array([ev["lam_mom_x"] for ev in events])
    py = np.array([ev["lam_mom_y"] for ev in events])
    pz = np.array([ev["lam_mom_z"] for ev in events])
    p = np.sqrt(px**2 + py**2 + pz**2)
    pt = np.sqrt(px**2 + py**2)
    return p, pt


def plot_training_history(results):
    plt.figure(figsize=(8, 5))
    for result in results:
        h = result["history"]
        epoch = [x["epoch"] for x in h]
        val_loss = [x["val_loss"] for x in h]
        plt.plot(epoch, val_loss, marker="o", markersize=3, label=result["label"])
    plt.xlabel("Epoch")
    plt.ylabel("Validation MSE on normalized z")
    plt.title("Validation loss")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


def plot_residual_overlay(evaluations, residual_range=(-15000, 15000), bins=80):
    plt.figure(figsize=(8, 5))
    for ev in evaluations:
        plt.hist(ev["residual"], bins=bins, range=residual_range, histtype="step", linewidth=1.8, label=ev["label"])
    plt.xlabel(r"$z_{pred} - z_{true}$ [mm]")
    plt.ylabel("Events")
    plt.title("Vertex-z residuals")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


def plot_pred_vs_true(ev):
    z_true = ev["z_true"]
    z_pred = ev["z_pred"]
    zmin = min(z_true.min(), z_pred.min())
    zmax = max(z_true.max(), z_pred.max())

    plt.figure(figsize=(6, 6))
    plt.hist2d(z_true, z_pred, bins=80, cmin=1)
    plt.plot([zmin, zmax], [zmin, zmax], linewidth=1.5)
    plt.xlabel("True z [mm]")
    plt.ylabel("Predicted z [mm]")
    plt.title(ev["label"])
    plt.colorbar(label="Events")
    plt.tight_layout()
    plt.show()


def plot_residual_vs_quantity(ev, quantity, xlabel, bins=(70, 70), residual_range=(-15000, 15000)):
    plt.figure(figsize=(7, 5))
    plt.hist2d(quantity, ev["residual"], bins=bins, range=[[np.min(quantity), np.max(quantity)], list(residual_range)], cmin=1)
    plt.axhline(0, linewidth=1.2)
    plt.xlabel(xlabel)
    plt.ylabel(r"$z_{pred} - z_{true}$ [mm]")
    plt.title(ev["label"])
    plt.colorbar(label="Events")
    plt.tight_layout()
    plt.show()

plot_training_history([wsi_result])
plot_residual_overlay(evaluations)

for ev in evaluations:
    plot_pred_vs_true(ev)

p_lam, pt_lam = lambda_momentum_arrays(test_events)
for ev in evaluations:
    plot_residual_vs_quantity(ev, ev["z_true"], "True z [mm]")
    plot_residual_vs_quantity(ev, p_lam, r"$\Lambda$ momentum [GeV]")
    plot_residual_vs_quantity(ev, pt_lam, r"$\Lambda$ transverse momentum [GeV]")


## Optional: save a compact PDF report

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages


def save_comparison_pdf(filename, evaluations, train_results, test_events):
    p_lam, pt_lam = lambda_momentum_arrays(test_events)

    with PdfPages(filename) as pdf:
        fig = plt.figure(figsize=(8, 5))
        for result in train_results:
            h = result["history"]
            plt.plot([x["epoch"] for x in h], [x["val_loss"] for x in h], marker="o", markersize=3, label=result["label"])
        plt.xlabel("Epoch")
        plt.ylabel("Validation MSE on normalized z")
        plt.title("Validation loss")
        plt.legend(frameon=False)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        fig = plt.figure(figsize=(8, 5))
        for ev in evaluations:
            plt.hist(ev["residual"], bins=80, range=(-15000, 15000), histtype="step", linewidth=1.8, label=ev["label"])
        plt.xlabel(r"$z_{pred} - z_{true}$ [mm]")
        plt.ylabel("Events")
        plt.title("Vertex-z residuals")
        plt.legend(frameon=False)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        for ev in evaluations:
            for quantity, xlabel in [
                (ev["z_true"], "True z [mm]"),
                (p_lam, r"$\Lambda$ momentum [GeV]"),
                (pt_lam, r"$\Lambda$ transverse momentum [GeV]"),
            ]:
                fig = plt.figure(figsize=(7, 5))
                plt.hist2d(quantity, ev["residual"], bins=(70, 70), cmin=1)
                plt.axhline(0, linewidth=1.2)
                plt.xlabel(xlabel)
                plt.ylabel(r"$z_{pred} - z_{true}$ [mm]")
                plt.title(ev["label"])
                plt.colorbar(label="Events")
                plt.tight_layout()
                pdf.savefig(fig)
                plt.close(fig)

save_comparison_pdf("wsi_vertex_report.pdf", evaluations, [wsi_result], test_events)
